# ⚡ Workshop 1: Building a Retrieval-Augmented Generation (RAG) System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AIMLCLUBOCT/Workshops/blob/main/advanced/notebooks/01_rag_retrieval_augmented_generation.ipynb)

**AI & Machine Learning Club – Oriental College of Technology, Bhopal**

Learn how modern Generative AI apps eliminate hallucinations using RAG:
1. Ingesting and chunking documents.
2. Dense vector embeddings with SentenceTransformers.
3. Semantic similarity search with Cosine Similarity.
4. Prompt grounding & synthesized answers with citations.

In [ ]:
# Install dependencies if running in Colab
# !pip install -q sentence-transformers
import numpy as np
from sentence_transformers import SentenceTransformer

print('Loading lightweight embedding model...')
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model loaded successfully!')

## 1. Document Knowledge Base (OCT Campus Regulations Demo)
Let us define our raw knowledge chunks about college academic ordinances.

In [ ]:
documents = [
    'Oriental College of Technology (OCT) was established in 2002 under the Oriental Group of Institutes in Bhopal.',
    'To be eligible for end-semester exams, students must maintain a minimum attendance rate of 75% in every course.',
    'AIML Club OCT organizes weekly workshops on PyTorch, Computer Vision, Generative AI, and autonomous AI agents.',
    'Students can access cloud GPU resources free of charge using Google Colab and the official club repository templates.',
    'Minor degree specializations in Artificial Intelligence and Machine Learning require 18 additional credits over 4 semesters.'
]

# Compute document embeddings (each becomes a 384-dimensional vector)
doc_embeddings = model.encode(documents, normalize_embeddings=True)
print('Document Embeddings shape:', doc_embeddings.shape)

## 2. Semantic Search & Top-K Retrieval
Given a student question, we encode the question and compute cosine similarity against all document vectors.

In [ ]:
def query_knowledge_base(query: str, top_k: int = 2):
    query_embedding = model.encode([query], normalize_embeddings=True)[0]
    # Cosine similarity is simply dot product because vectors are normalized
    scores = np.dot(doc_embeddings, query_embedding)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    print(f'🔍 Query: "{query}"\n')
    for rank, idx in enumerate(top_indices, 1):
        print(f'Rank {rank} (Confidence: {scores[idx]:.3f}):\n  "{documents[idx]}"\n')

# Test semantic search with student questions
query_knowledge_base('What is the minimum attendance required to appear in exams?')
query_knowledge_base('Where can I learn PyTorch and Generative AI at OCT?')

## 3. Grounded Prompt Formulation
Now we construct the grounded prompt that is fed into an LLM (e.g. Llama 3 or Gemini) to prevent hallucination.

In [ ]:
def build_rag_prompt(user_question: str, retrieved_docs: list) -> str:
    context_block = '\n'.join([f'- {doc}' for doc in retrieved_docs])
    prompt = f'''You are the official AI Assistant for Oriental College of Technology (OCT).
Answer the question strictly using the provided context below. Do not make up information.

CONTEXT:
{context_block}

STUDENT QUESTION:
{user_question}

ANSWER:'''
    return prompt

sample_prompt = build_rag_prompt(
    'How much attendance do I need for exams?',
    [documents[1]]
)
print(sample_prompt)